# Question-1: self-driving car

1. Performance Measure	
    1) Efficiency: Use the fewest moves possible.
    2) Safety: Don't crash while moving.
2. Environment	
    1) Road: Two Intersection A and B on a road.
    2) Traffic: Either "Clear" or "Congested".
3. Actuators	
    1) Steering Wheel: To guide the direction to movement.
    2) Signal System: To perform "Wait/Clear Traffic" and signal the resolution.
4. Sensors	
    1) Location Sensor: To know if it’s at A or B.
    2) Traffic Sensor: To see if a spot is congested.

In [ ]:
# This function helps in calculating a list of all possible (next_state, action_name) from current state
def get_moves(state):
    loc, a_congested, b_congested = state
    moves = []

    # Action: Move
    if loc == 'A':
        moves.append((('B', a_congested, b_congested), "Move Right"))
    else:
        moves.append((('A', a_congested, b_congested), "Move Left"))
        
    # Action: Clear Traffic (only if current spot is congested)
    if loc == 'A' and a_congested == 1:
        moves.append((('A', 0, b_congested), "Clear A"))
    elif loc == 'B' and b_congested == 1:
        moves.append((('B', a_congested, 0), "Clear B"))
        
    return moves

def solve(start_state, method="BFS"):
    # Each item in the frontier is: [current_state, path_taken_so_far]
    frontier = [[start_state, []]]
    explored = []

    while frontier:
        # BFS: Pop the first item (index 0) | DFS: Pop the last item (no index)
        current_node = frontier.pop(0) if method == "BFS" else frontier.pop()
        state, path = current_node

        if state in explored:
            continue
        explored.append(state)

        # Check if traffic is clear (0, 0)
        if state[1] == 0 and state[2] == 0:
            return path + [(state, "Goal")]

        # Explore neighbors
        for next_state, action in get_moves(state):
            if next_state not in explored:
                new_path = path + [(state, action)]
                frontier.append([next_state, new_path])

start = ('A', 1, 1)

for mode in ["BFS", "DFS"]:
    print(f"\n{mode} Solution")
    solution = solve(start, mode)
    for state, action in solution:
        print(f"State: {state} -> Action: {action}")


BFS Solution
State: ('A', 1, 1) -> Action: Clear A
State: ('A', 0, 1) -> Action: Move Right
State: ('B', 0, 1) -> Action: Clear B
State: ('B', 0, 0) -> Action: Goal

DFS Solution
State: ('A', 1, 1) -> Action: Clear A
State: ('A', 0, 1) -> Action: Move Right
State: ('B', 0, 1) -> Action: Clear B
State: ('B', 0, 0) -> Action: Goal


| Feature | **Breadth-First Search (BFS)** | **Depth-First Search (DFS)** |
| --- | --- | --- |
| **Max States Generated** | **Higher.** Explores every possible state at the current "depth" before moving deeper. | **Variable.** Can be lower if it hits the goal early, but often explores redundant paths first. |
| **Completeness** | **Yes.** It is guaranteed to find the goal in a finite state space like this one. | **Yes.** (Since we use a "visited" list to prevent the car from driving in circles forever). |
| **Optimality** | **Yes.** Since every action cost is , BFS always finds the shortest path. | **No.** It will lead to goal state only after making unnecessary moves. |

# Question-2: N-Puzzle Solver

In [ ]:
import heapq

# I am representing these in tuple format for easiness in indexing
# where weach index is indexed by // (integer division) and % (modulus)
START_LAYOUT = (1, 2, 3, 
                8, 0, 4, 
                7, 6, 5)

TARGET_LAYOUT = (2, 8, 1, 
                 0, 4, 3, 
                 7, 6, 5)

# This function helps in calculating Manhattan distance (heuristic)
def calc_estimate(current_box):
    dist_total = 0
    for i in range(9):
        val = current_box[i]
        if val != 0:
            # Find where the tile is now vs where it should be
            now_r, now_c = i // 3, i % 3
            goal_idx = TARGET_LAYOUT.index(val)
            goal_r, goal_c = goal_idx // 3, goal_idx % 3
            dist_total += abs(now_r - goal_r) + abs(now_c - goal_c)
    return dist_total

# This function helps in finding all valid adjacent states by swapping the empty spot (0)
def get_possible_moves(box):
    results = []
    empty_idx = box.index(0)
    r, c = empty_idx // 3, empty_idx % 3
    
    shifts = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]
    
    for dr, dc, move_name in shifts:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_idx = nr * 3 + nc
            # Swap the empty spot with the tile
            temp_list = list(box)
            temp_list[empty_idx], temp_list[new_idx] = temp_list[new_idx], temp_list[empty_idx]
            results.append((tuple(temp_list), move_name))
    return results

def run_puzzle_solver():
    # Priority queue stores: (total_score, steps_taken, current_state, history)
    # total_score = g(n) + h(x)
    work_queue = [(calc_estimate(START_LAYOUT), 0, START_LAYOUT, [])]
    seen_states = {START_LAYOUT: 0}
    
    while work_queue:
        _, cost, current, log = heapq.heappop(work_queue)
        
        if current == TARGET_LAYOUT:
            return log, current
            
        for neighbor, move in get_possible_moves(current):
            new_cost = cost + 1
            # If we haven't seen this layout or found a faster way to it
            if neighbor not in seen_states or new_cost < seen_states[neighbor]:
                seen_states[neighbor] = new_cost
                total_priority = new_cost + calc_estimate(neighbor)
                heapq.heappush(work_queue, (total_priority, new_cost, neighbor, log + [(neighbor, move)]))

path_found, final_state = run_puzzle_solver()

print("Initial State:", START_LAYOUT[:3], START_LAYOUT[3:6], START_LAYOUT[6:], sep="\n")
print("-" * 20)

for step_state, move_dir in path_found:
    print(f"Action: {move_dir}")
    print(f"State: {step_state[:3]}")
    print(f"       {step_state[3:6]}")
    print(f"       {step_state[6:]}")
    print("-" * 20)

print("Goal Reached!")

Initial State:
(1, 2, 3)
(8, 0, 4)
(7, 6, 5)
--------------------
Action: Up
State: (1, 0, 3)
       (8, 2, 4)
       (7, 6, 5)
--------------------
Action: Left
State: (0, 1, 3)
       (8, 2, 4)
       (7, 6, 5)
--------------------
Action: Down
State: (8, 1, 3)
       (0, 2, 4)
       (7, 6, 5)
--------------------
Action: Right
State: (8, 1, 3)
       (2, 0, 4)
       (7, 6, 5)
--------------------
Action: Right
State: (8, 1, 3)
       (2, 4, 0)
       (7, 6, 5)
--------------------
Action: Up
State: (8, 1, 0)
       (2, 4, 3)
       (7, 6, 5)
--------------------
Action: Left
State: (8, 0, 1)
       (2, 4, 3)
       (7, 6, 5)
--------------------
Action: Left
State: (0, 8, 1)
       (2, 4, 3)
       (7, 6, 5)
--------------------
Action: Down
State: (2, 8, 1)
       (0, 4, 3)
       (7, 6, 5)
--------------------
Goal Reached!


# Task-3: Uniform Cost Search 

In [10]:
import heapq # library for priority queue

# I the graph as a dictionary: {Start: {End: Cost}}
map_network = {
    'A': {'B': 32, 'F': 3},
    'B': {'E': 12, 'C': 21},
    'F': {'B': 7, 'C': 2},
    'E': {'D': 13, 'C': 6},
    'C': {'G': 11},
    'D': {'G': 9},
    'G': {}
}

def run_ucs(start_node, goal_node):
    # todo_list stores (total_cost, current_node, path_history)
    todo_list = [(0, start_node, [])]
    visited_nodes = {}

    while todo_list:
        # Picking the cheapest path found so far
        cost, current, path = heapq.heappop(todo_list)

        if current in visited_nodes and visited_nodes[current] <= cost:
            continue
        
        visited_nodes[current] = cost
        new_path = path + [current]

        if current == goal_node:
            return new_path, cost

        # Checking all connected neighbors
        for neighbor, edge_price in map_network.get(current, {}).items():
            total_price = cost + edge_price
            heapq.heappush(todo_list, (total_price, neighbor, new_path))

    return None, 0

# Running the search
final_route, final_cost = run_ucs('A', 'G')
print(f"Optimal Path: {' -> '.join(final_route)}")
print(f"Total Path Cost: {final_cost}")

Optimal Path: A -> F -> C -> G
Total Path Cost: 16
